In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))

from src.config import CFG
import src.functions as fn
from src.functions import (
    keep_from_peak,
    length_fix,
    split_and_add,
    group_by_activity,
    group_by_prefix,
)

In [0]:
# Paths
VOLUME_PATH = CFG["data"]["volume_path"]

print("Config loaded ✅")
print(f"Volume path : {VOLUME_PATH}")
print(f"Sample rate : {CFG['data']['sample_rate']} Hz")
print(f"Target length : {CFG['data']['signal_length']} samples")

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Load all arrays from volume
X_all              = np.load(f"{VOLUME_PATH}/X_all.npy",              allow_pickle=True)
y_all              = np.load(f"{VOLUME_PATH}/y_all.npy",              allow_pickle=True)
activity_codes_all = np.load(f"{VOLUME_PATH}/activity_codes_all.npy", allow_pickle=True)
file_names_all     = np.load(f"{VOLUME_PATH}/file_names_all.npy",     allow_pickle=True)

print(f"Total signals    : {X_all.shape[0]}")
print(f"Alignment check  : ", end="")

# Verify all 4505 signals are aligned
mismatches = sum(
    1 for i in range(len(file_names_all))
    if file_names_all[i].split("_")[0] != activity_codes_all[i]
)
print(f"✅ Perfect" if mismatches == 0 else f"❌ {mismatches} mismatches")

# Split into train and test based on config
TRAIN_SUBJECTS = CFG["data"]["train_subjects"]
TEST_SUBJECTS  = CFG["data"]["test_subjects"]

train_idx = [i for i, f in enumerate(file_names_all)
             if any(sub in f for sub in TRAIN_SUBJECTS)]
test_idx  = [i for i, f in enumerate(file_names_all)
             if any(sub in f for sub in TEST_SUBJECTS)]

X_train              = X_all[train_idx]
y_train              = y_all[train_idx]
activity_codes_train = activity_codes_all[train_idx]
file_names_train     = file_names_all[train_idx]

X_test               = X_all[test_idx]
y_test               = y_all[test_idx]
activity_codes_test  = activity_codes_all[test_idx]
file_names_test      = file_names_all[test_idx]

print(f"Train : {X_train.shape[0]} signals  "
      f"ADL={np.sum(y_train=='ADL')}  Fall={np.sum(y_train=='Fall')}")
print(f"Test  : {X_test.shape[0]} signals   "
      f"ADL={np.sum(y_test=='ADL')}  Fall={np.sum(y_test=='Fall')}")

In [0]:
# Split X_train into activity groups
# Falls — all F codes together
fall_data = group_by_prefix(X_train, activity_codes_train, "F")

# ADL groups
d01_data = group_by_activity(X_train, activity_codes_train, "D01")
d02_data = group_by_activity(X_train, activity_codes_train, "D02")
d03_data = group_by_activity(X_train, activity_codes_train, "D03")
d04_data = group_by_activity(X_train, activity_codes_train, "D04")
d05_data = group_by_activity(X_train, activity_codes_train, "D05")
d06_data = group_by_activity(X_train, activity_codes_train, "D06")
d07_data = group_by_activity(X_train, activity_codes_train, "D07")
d08_data = group_by_activity(X_train, activity_codes_train, "D08")
d09_data = group_by_activity(X_train, activity_codes_train, "D09")
d10_data = group_by_activity(X_train, activity_codes_train, "D10")
d11_data = group_by_activity(X_train, activity_codes_train, "D11")
d12_data = group_by_activity(X_train, activity_codes_train, "D12")
d13_data = group_by_activity(X_train, activity_codes_train, "D13")
d14_data = group_by_activity(X_train, activity_codes_train, "D14")
d15_data = group_by_activity(X_train, activity_codes_train, "D15")
d16_data = group_by_activity(X_train, activity_codes_train, "D16")
d17_data = group_by_activity(X_train, activity_codes_train, "D17")
d18_data = group_by_activity(X_train, activity_codes_train, "D18")
d19_data = group_by_activity(X_train, activity_codes_train, "D19")

# Group file names alongside signals
d01_names = group_by_activity(file_names_train, activity_codes_train, "D01")
d02_names = group_by_activity(file_names_train, activity_codes_train, "D02")
d03_names = group_by_activity(file_names_train, activity_codes_train, "D03")
d04_names = group_by_activity(file_names_train, activity_codes_train, "D04")

# Quick summary
print(f"\nGroup sizes:")
print(f"  Falls        : {len(fall_data)}")
print(f"  D01 (walk slow)  : {len(d01_data)}")
print(f"  D02 (walk fast)  : {len(d02_data)}")
print(f"  D03 (jog slow)   : {len(d03_data)}")
print(f"  D04 (jog fast)   : {len(d04_data)}")
print(f"  D05 (stairs slow): {len(d05_data)}")
print(f"  D06 (stairs fast): {len(d06_data)}")
print(f"  D07–D19          : {sum([len(d07_data), len(d08_data), len(d09_data), len(d10_data), len(d11_data), len(d12_data), len(d13_data), len(d14_data), len(d15_data), len(d16_data), len(d17_data), len(d18_data), len(d19_data)])}")

In [0]:
print("Processing Falls...")

# Step 1 — find peak and keep 400 samples each side
fall_peaked = keep_from_peak(fall_data, window_size=400)

# Step 2 — guarantee exactly 800 samples for every signal
fall_processed = length_fix(fall_peaked, length=800)

# Step 3 — verify
shapes = set(s.shape for s in fall_processed)
print(f"\nFall signals processed : {len(fall_processed)}")
print(f"Unique shapes          : {shapes}")
print(f"Expected               : {{(6, 800)}}")
print(f"All correct            : {shapes == {(6, 800)}}")

In [0]:
print("Processing D01–D04 (walking/jogging)...")

# split_and_add returns (np.array, file_names, activity_codes)
new_d01, _, d01_codes = split_and_add(d01_data, 800, d01_names)
new_d02, _, d02_codes = split_and_add(d02_data, 800, d02_names)
new_d03, _, d03_codes = split_and_add(d03_data, 800, d03_names)
new_d04, _, d04_codes = split_and_add(d04_data, 800, d04_names)

# Verify shapes
for name, data in [("D01", new_d01), ("D02", new_d02),
                   ("D03", new_d03), ("D04", new_d04)]:
    shapes = set(s.shape for s in data)
    print(f"  {name} → {len(data)} windows  shapes={shapes}")

In [0]:
print("Processing D05 (stairs slowly)...")
d05_cleaned        = idle_remover(d05_data, window_size=200, scale=30, mode='gyro')
d05_seg1, d05_seg2 = split_and_center(d05_cleaned, length=800)
d05_proc1          = length_fix(d05_seg1, length=800)
d05_proc2          = length_fix(d05_seg2, length=800)

# Verify
shapes1 = set(s.shape for s in d05_proc1)
shapes2 = set(s.shape for s in d05_proc2)
print(f"\nD05 segment 1 → {len(d05_proc1)} signals  shapes={shapes1}")
print(f"D05 segment 2 → {len(d05_proc2)} signals  shapes={shapes2}")

In [0]:
print("Processing D06 (stairs quickly)...")
d06_cleaned        = idle_remover(d06_data, window_size=150, scale=30, mode='acc')
d06_seg1, d06_seg2 = split_and_center(d06_cleaned, length=800)
d06_proc1          = length_fix(d06_seg1, length=800)
d06_proc2          = length_fix(d06_seg2, length=800)

# Verify
shapes1 = set(s.shape for s in d06_proc1)
shapes2 = set(s.shape for s in d06_proc2)
print(f"\nD06 segment 1 → {len(d06_proc1)} signals  shapes={shapes1}")
print(f"D06 segment 2 → {len(d06_proc2)} signals  shapes={shapes2}")

In [0]:
from src.functions import extract_from_high_amp_segments

print("Processing D07 (Slowly sit in a half height chair, wait a moment, and up slowly) ...")

d07_seg1, d07_seg2 = extract_from_high_amp_segments(
    d07_data,
    window_size=100,
    step_size=100,
    amp_scale=0.3,
    min_gap=700,
    before=200,
    after=800,
    sensor_type="gyro"
)
d07_proc1          = length_fix(d07_seg1, length=800)
d07_proc2          = length_fix(d07_seg2, length=800)

# Verify
shapes1 = set(s.shape for s in d07_proc1)
shapes2 = set(s.shape for s in d07_proc2)
print(f"\nD07 segment 1 → {len(d07_proc1)} signals  shapes={shapes1}")
print(f"D07 segment 2 → {len(d07_proc2)} signals  shapes={shapes2}")

In [0]:
print("Processing D08 (Quickly sit in a half height chair, wait a moment, and up quickly)...")

d08_seg1, d08_seg2 = extract_from_high_amp_segments(
    d08_data,
    window_size=100,
    step_size=100,
    amp_scale=0.14,
    min_gap=500,
    before=100,
    after=500,
    sensor_type="gyro"
)
d08_proc1          = length_fix(d08_seg1, length=800)
d08_proc2          = length_fix(d08_seg2, length=800)

# Verify
shapes1 = set(s.shape for s in d08_proc1)
shapes2 = set(s.shape for s in d08_proc2)
print(f"\nD08 segment 1 → {len(d08_proc1)} signals  shapes={shapes1}")
print(f"D08 segment 2 → {len(d08_proc2)} signals  shapes={shapes2}")

In [0]:
print("Processing D09 (Slowly sit in a low height chair, wait a moment, and up slowly)...")

d09_seg1, d09_seg2 = extract_from_high_amp_segments(
    d09_data,
    window_size=100,
    step_size=100,
    amp_scale=0.1,
    min_gap=700,
    before=100,
    after=700,
    sensor_type="gyro"
)
d09_proc1          = length_fix(d09_seg1, length=800)
d09_proc2          = length_fix(d09_seg2, length=800)

# Verify
shapes1 = set(s.shape for s in d09_proc1)
shapes2 = set(s.shape for s in d09_proc2)
print(f"\nD09 segment 1 → {len(d09_proc1)} signals  shapes={shapes1}")
print(f"D09 segment 2 → {len(d09_proc2)} signals  shapes={shapes2}")

In [0]:
print("Processing D10 (Quickly sit in a low height chair, wait a moment, and up quickly)...")

d10_seg1, d10_seg2 = extract_from_high_amp_segments(
    d10_data,
    window_size=100,
    step_size=100,
    amp_scale=0.1,
    min_gap=500,
    before=100,
    after=500,
    sensor_type="gyro"
)
d10_proc1          = length_fix(d10_seg1, length=800)
d10_proc2          = length_fix(d10_seg2, length=800)

# Verify
shapes1 = set(s.shape for s in d10_proc1)
shapes2 = set(s.shape for s in d10_proc2)
print(f"\nD10 segment 1 → {len(d10_proc1)} signals  shapes={shapes1}")
print(f"D10 segment 2 → {len(d10_proc2)} signals  shapes={shapes2}")

In [0]:
print("Processing D11 (Sitting a moment, trying to get up, and collapse into a chair)...")

# Step 1 — find peak and keep 400 samples each side
d11_cleaned = keep_from_peak(d11_data, window_size=400)

# Step 2 — guarantee exactly 800 samples for every signal
d11_proc = length_fix(d11_cleaned, length=800)

# Step 3 — verify
shapes = set(s.shape for s in d11_proc)
print(f"\nD11 signals processed : {len(d11_proc)}")
print(f"Unique shapes          : {shapes}")
print(f"Expected               : {{(6, 800)}}")
print(f"All correct            : {shapes == {(6, 800)}}")

In [0]:
print("Processing D12 (Sitting a moment, lying slowly, wait a moment, and sit again)...")

d12_seg1, d12_seg2 = extract_from_high_amp_segments(
    d12_data,
    window_size=100,
    step_size=100,
    amp_scale=0.2,
    min_gap=800,
    before=200,
    after=800,
    sensor_type="gyro"
)
d12_proc1          = length_fix(d12_seg1, length=800)
d12_proc2          = length_fix(d12_seg2, length=800)

# Verify
shapes1 = set(s.shape for s in d12_proc1)
shapes2 = set(s.shape for s in d12_proc2)
print(f"\nD12 segment 1 → {len(d12_proc1)} signals  shapes={shapes1}")
print(f"D12 segment 2 → {len(d12_proc2)} signals  shapes={shapes2}")

In [0]:
print("Processing D13 (Sitting a moment, lying quickly, wait a moment, and sit again)...")

d13_seg1, d13_seg2 = extract_from_high_amp_segments(
    d13_data,
    window_size=100,
    step_size=100,
    amp_scale=0.2,
    min_gap=500,
    before=50,
    after=500,
    sensor_type="gyro"
)
d13_proc1          = length_fix(d13_seg1, length=800)
d13_proc2          = length_fix(d13_seg2, length=800)

# Verify
shapes1 = set(s.shape for s in d13_proc1)
shapes2 = set(s.shape for s in d13_proc2)
print(f"\nD13 segment 1 → {len(d13_proc1)} signals  shapes={shapes1}")
print(f"D13 segment 2 → {len(d13_proc2)} signals  shapes={shapes2}")

In [0]:
print("Processing D14 (Being on one's back change to lateral position, wait a moment, and change to one's back)...")

d14_seg1, d14_seg2 = extract_from_high_amp_segments(
    d14_data,
    window_size=100,
    step_size=100,
    amp_scale=0.15,
    min_gap=600,
    before=100,
    after=600,
    sensor_type="gyro"
)
d14_proc1          = length_fix(d14_seg1, length=800)
d14_proc2          = length_fix(d14_seg2, length=800)

# Verify
shapes1 = set(s.shape for s in d14_proc1)
shapes2 = set(s.shape for s in d14_proc2)
print(f"\nD14 segment 1 → {len(d14_proc1)} signals  shapes={shapes1}")
print(f"D14 segment 2 → {len(d14_proc2)} signals  shapes={shapes2}")

In [0]:
print("Processing D15 (Standing, slowly bending at knees, and getting up)...")

d15_seg1, d15_seg2 = extract_from_high_amp_segments(
    d15_data,
    window_size=250,
    step_size=250,
    amp_scale=0.2,
    min_gap=700,
    before=200,
    after=800,
    sensor_type="gyro"
)
d15_proc1          = length_fix(d15_seg1, length=800)
d15_proc2          = length_fix(d15_seg2, length=800)

# Verify
shapes1 = set(s.shape for s in d15_proc1)
shapes2 = set(s.shape for s in d15_proc2)
print(f"\nD15 segment 1 → {len(d15_proc1)} signals  shapes={shapes1}")
print(f"D15 segment 2 → {len(d15_proc2)} signals  shapes={shapes2}")

In [0]:
print("Processing D16 (Standing, slowly bending without bending knees, and getting up)...")

d16_seg1, d16_seg2 = extract_from_high_amp_segments(
    d16_data,
    window_size=100,
    step_size=100,
    amp_scale=0.15,
    min_gap=700,
    before=200,
    after=800,
    sensor_type="gyro"
)
d16_proc1          = length_fix(d16_seg1, length=800)
d16_proc2          = length_fix(d16_seg2, length=800)

# Verify
shapes1 = set(s.shape for s in d16_proc1)
shapes2 = set(s.shape for s in d16_proc2)
print(f"\nD16 segment 1 → {len(d16_proc1)} signals  shapes={shapes1}")
print(f"D16 segment 2 → {len(d16_proc2)} signals  shapes={shapes2}")

In [0]:
print("Processing D17 (Standing, get into a car, remain seated and get out of the car)...")

# trim data to remove the first 300 samples
for i in range(len(d17_data)):
    d17_data[i] = d17_data[i][:, 300:]

d17_seg1, d17_seg2 = extract_from_high_amp_segments(
    d17_data,
    window_size=300,
    step_size=300,
    amp_scale=0.3,
    min_gap=1000,
    before=200,
    after=1000,
    sensor_type="gyro"
)
d17_proc1          = length_fix(d17_seg1, length=1000)
d17_proc2          = length_fix(d17_seg2, length=1000)

# Verify
shapes1 = set(s.shape for s in d17_proc1)
shapes2 = set(s.shape for s in d17_proc2)
print(f"\nD17 segment 1 → {len(d17_proc1)} signals  shapes={shapes1}")
print(f"D17 segment 2 → {len(d17_proc2)} signals  shapes={shapes2}")

In [0]:
print("Processing D18 (Stumble while walking)...")

# Step 1 — find peak and keep 400 samples each side
d18_cleaned = keep_from_peak(d18_data, window_size=400)

# Step 2 — guarantee exactly 800 samples for every signal
d18_proc = length_fix(d18_cleaned, length=800)

# Step 3 — verify
shapes = set(s.shape for s in d18_proc)
print(f"\nD18 signals processed : {len(d18_proc)}")
print(f"Unique shapes          : {shapes}")
print(f"Expected               : {{(6, 800)}}")
print(f"All correct            : {shapes == {(6, 800)}}")

In [0]:
print("Processing D19 (Gently jump without falling (trying to reach a high object))...")

d19_seg1, d19_seg2 = extract_from_high_amp_segments(
    d19_data,
    window_size=100,
    step_size=100,
    amp_scale=0.12,
    min_gap=400,
    before=150,
    after=500,
    sensor_type="acc"
)
d19_proc1          = length_fix(d19_seg1, length=800)
d19_proc2          = length_fix(d19_seg2, length=800)

# Verify
shapes1 = set(s.shape for s in d19_proc1)
shapes2 = set(s.shape for s in d19_proc2)
print(f"\nD19 segment 1 → {len(d19_proc1)} signals  shapes={shapes1}")
print(f"D19 segment 2 → {len(d19_proc2)} signals  shapes={shapes2}")